# 08 - Uncertainty and business checks

Build approximate P10/P50/P90 forecast bands from the validated-blend rolling-origin residuals, and report leave-one-origin-out empirical coverage. Aggregate forecasts by month, family, and region; P10/P90 rollups are sums of SKU-level bounds, not aggregate-level quantiles. Assert that forecasts and intervals are zero before launch and after the inclusive EOL month; known events are exposed without unsupported demand effects.

The source data do not include on-hand inventory, open purchase orders, lead times, or service-level targets. This notebook therefore reports demand and uncertainty, not unsupported reorder quantities.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "outputs" / "submission").is_dir():
            return candidate
    raise FileNotFoundError("Run notebook 07 before the business checks.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from src.business_checks import run_business_checks

summary = run_business_checks(PROJECT_ROOT)
summary

In [ ]:
import pandas as pd
from IPython.display import display

output_dir = PROJECT_ROOT / "outputs" / "business_checks"
quantiles = pd.read_csv(output_dir / "forecast_quantiles.csv", low_memory=False)
monthly = pd.read_csv(output_dir / "monthly_total_forecast.csv")
by_family = pd.read_csv(output_dir / "monthly_forecast_by_family.csv")
by_region = pd.read_csv(output_dir / "monthly_forecast_by_region.csv")
coverage = pd.read_csv(output_dir / "interval_leave_one_origin_out_coverage.csv")

display(monthly)
display(coverage)
display(by_family.head(12))
display(by_region.head(12))
display(quantiles[["Date", "SKU_ID", "Region", "Forecast_Method", "P10", "P50", "P90"]].head())

assert summary["forecast_rows"] == 58_219
assert summary["forecast_months"] == 19
assert summary["hierarchy_reconciles"]
assert quantiles[["P10", "P50", "P90"]].notna().all().all()
assert (quantiles["P10"] <= quantiles["P50"]).all()
assert (quantiles["P50"] <= quantiles["P90"]).all()
assert set(coverage["Origin"]) == {"2023-09", "2025-03"}
print("Validation passed: complete quantiles, reconciled family/region totals, and both held-out origins reported.")

Treat the interval coverage and lifecycle/event exposures as decision support, not guarantees. A purchasing recommendation still needs current inventory, committed inbound units, supplier lead times, and a chosen service level.